# Skin disease classification: full experiment on Kaggle

This notebook runs `scripts/run_experiment.py` and `scripts/analyze.py` from this repository. It produces every number, table and figure in the paper.

**Before running**

1. *Settings → Accelerator*: **GPU T4 x2** (the P100 is retired). *Settings → Internet*: **On** (ImageNet weights are downloaded).
2. *Add Input → Datasets*: the skin disease images. Use the original Mendeley release (doi:10.17632/3hckgznc67.1, 9,548 images) uploaded as a Kaggle dataset. Then set `DATA_ROOTS` below to the folder(s) that contain one sub-folder per class.
3. Get the code: either the repository is public (it is cloned), or upload the repository as a Kaggle dataset named `skinclf-code` and add it as input.
4. *Save Version → Save & Run All (Commit)*, so the run continues with the browser closed.

**If a session ends early**, every finished (model, fold) is kept. Add this notebook's previous output as an input (*Add Input → Notebook Output*), set `PREVIOUS_OUTPUT` to its `outputs` folder, and commit again. Finished runs are skipped.

In [ ]:
import os, shutil, subprocess, glob

DATA_ROOTS = ["/kaggle/input/skin-disease-classification-dataset"]  # one or more folders of <class>/<images>
CODE_REPO = "https://github.com/farzanasupti/skin-disease-classification.git"
PREVIOUS_OUTPUT = None  # e.g. "/kaggle/input/skinclf-run-1/outputs" when resuming
HOURS = 11.0            # stop starting new runs after this (Kaggle sessions end at 12 h)
OUT = "/kaggle/working/outputs"

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

uploaded = glob.glob("/kaggle/input/skinclf-code*/**/scripts/run_experiment.py", recursive=True)
if uploaded:
    CODE = os.path.dirname(os.path.dirname(uploaded[0]))
else:
    subprocess.run(["git", "clone", "--depth", "1", CODE_REPO, "/kaggle/working/code"], check=True)
    CODE = "/kaggle/working/code"
print("code:", CODE)
print(subprocess.run(["git", "-C", CODE, "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

def find_data_roots(roots):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, on newer images, deeper (e.g. /kaggle/input/datasets/<owner>/<slug>)."""
    found = []
    for r in roots:
        if os.path.isdir(r):
            found.append(r)
            continue
        hits = sorted(p for p in glob.glob(f"/kaggle/input/**/{os.path.basename(r.rstrip('/'))}", recursive=True) if os.path.isdir(p))
        if not hits:
            tree = [p for p in glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*") + glob.glob("/kaggle/input/*/*/*") if os.path.isdir(p)]
            raise FileNotFoundError(f"{r} not found. Folders under /kaggle/input: {tree[:40]}")
        found.append(hits[0])
    return found

DATA_ROOTS = find_data_roots(DATA_ROOTS)
print("data:", DATA_ROOTS)

if PREVIOUS_OUTPUT:
    PREVIOUS_OUTPUT = find_data_roots([PREVIOUS_OUTPUT])[0]
if PREVIOUS_OUTPUT and not os.path.exists(OUT):
    shutil.copytree(PREVIOUS_OUTPUT, OUT)
    print("resumed from", PREVIOUS_OUTPUT, "with", len(glob.glob(OUT + "/runs/*/summary.json")), "finished runs")

In [ ]:
# Look at the data before training: class folders, counts and a few file names.
for root in DATA_ROOTS:
    for cls in sorted(os.listdir(root)):
        d = os.path.join(root, cls)
        if os.path.isdir(d):
            files = sorted(os.listdir(d))
            print(f"{cls:20s} {len(files):5d}  e.g. {files[:3]}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Mixed precision speeds up T4 GPUs. Drop --mixed-precision on P100 if anything looks off.
!python -u {CODE}/scripts/run_experiment.py --data-roots {" ".join(DATA_ROOTS)} --out-dir {OUT} --mixed-precision --hours {HOURS} 2>&1 | grep -v "^Epoch [0-9]*/" 

In [ ]:
!cat {OUT}/grouping_stats.json; echo; cat {OUT}/split_counts.csv
!ls {OUT}/runs | wc -l

In [ ]:
# Runs only when every model has every fold. Otherwise commit again with PREVIOUS_OUTPUT set.
!python {CODE}/scripts/analyze.py --out-dir {OUT} --paper-dir {OUT}/paper_generated

In [ ]:
shutil.make_archive("/kaggle/working/skinclf_outputs", "zip", OUT)
print("Download skinclf_outputs.zip from the Output tab.")